In [1]:
import numpy as np
import time , os 

In [5]:
#Allocate an 3*2 matris in ram called a 
a = np.zeros((3,2))
a.shape

(3, 2)

### 📝 NumPy Veri Oluşturma Rehberi

| Komut | Amacı | Boyut (Shape) | Tip |
| :--- | :--- | :---: | :--- |
| `np.zeros((3,2))` | Bellekte yer allocate eder, içini **0.0** ile doldurur. | $3 \times 2$ | Matris |
| `np.ones((5,))` | Belirtilen sayıda **1.0** değeri oluşturur. | $5$ | Vektör |
| `np.arange(0,10)` | Belirli aralıktaki sayıları **sırayla** dizer. | $10$ | Vektör |
| `np.random.rand(3,3)`| 0-1 arası **rastgele** sayılarla matris oluşturur. | $3 \times 3$ | Matris |

> **💡 Mühendis Notu:** > NumPy array'leri RAM'de bitişik (contiguous) bloklar halinde tutulur. Bu sayede `for` döngüsüne ihtiyaç duymadan, işlemcinin SIMD birimlerini veya RTX 4050'nin CUDA çekirdeklerini kullanarak "Vektörizasyon" yapabiliriz.

In [15]:
b = np.random.rand(3,2)
c = np.ones((3,2))
b,c

(array([[0.8112529 , 0.10245759],
        [0.24056345, 0.49095759],
        [0.84077792, 0.36550646]]),
 array([[1., 1.],
        [1., 1.],
        [1., 1.]]))

In [16]:
d = np.dot(b,c)
d

ValueError: shapes (3,2) and (3,2) not aligned: 2 (dim 1) != 3 (dim 0)

In [19]:
c = c.transpose()
c.shape

(2, 3)

In [21]:
d = np.dot(b,c)
d

array([[0.91371049, 0.91371049, 0.91371049],
       [0.73152104, 0.73152104, 0.73152104],
       [1.20628438, 1.20628438, 1.20628438]])

In [22]:
X = np.array([[1],[2],[3],[4]])
w = np.array([2])
c = np.dot(X[1], w)

print(f"X[1] has shape {X[1].shape}")
print(f"w has shape {w.shape}")
print(f"c has shape {c.shape}")

X[1] has shape (1,)
w has shape (1,)
c has shape ()


In [23]:
#vector 2-D slicing operations
a = np.arange(20).reshape(-1, 10)
print(f"a = \n{a}")

#access 5 consecutive elements (start:stop:step)
print("a[0, 2:7:1] = ", a[0, 2:7:1], ",  a[0, 2:7:1].shape =", a[0, 2:7:1].shape, "a 1-D array")

#access 5 consecutive elements (start:stop:step) in two rows
print("a[:, 2:7:1] = \n", a[:, 2:7:1], ",  a[:, 2:7:1].shape =", a[:, 2:7:1].shape, "a 2-D array")

# access all elements
print("a[:,:] = \n", a[:,:], ",  a[:,:].shape =", a[:,:].shape)

# access all elements in one row (very common usage)
print("a[1,:] = ", a[1,:], ",  a[1,:].shape =", a[1,:].shape, "a 1-D array")
# same as
print("a[1]   = ", a[1],   ",  a[1].shape   =", a[1].shape, "a 1-D array")

a = 
[[ 0  1  2  3  4  5  6  7  8  9]
 [10 11 12 13 14 15 16 17 18 19]]
a[0, 2:7:1] =  [2 3 4 5 6] ,  a[0, 2:7:1].shape = (5,) a 1-D array
a[:, 2:7:1] = 
 [[ 2  3  4  5  6]
 [12 13 14 15 16]] ,  a[:, 2:7:1].shape = (2, 5) a 2-D array
a[:,:] = 
 [[ 0  1  2  3  4  5  6  7  8  9]
 [10 11 12 13 14 15 16 17 18 19]] ,  a[:,:].shape = (2, 10)
a[1,:] =  [10 11 12 13 14 15 16 17 18 19] ,  a[1,:].shape = (10,) a 1-D array
a[1]   =  [10 11 12 13 14 15 16 17 18 19] ,  a[1].shape   = (10,) a 1-D array


In [1]:
import copy, math
import numpy as np
import matplotlib.pyplot as plt

# 2 Problem Statement

You will use the motivating example of housing price prediction. The training dataset contains three examples with four features (size, bedrooms, floors and, age) shown in the table below.  Note that, unlike the earlier labs, size is in sqft rather than 1000 sqft. This causes an issue, which you will solve in the next lab!

| Size (sqft) | Number of Bedrooms  | Number of floors | Age of  Home | Price (1000s dollars)  |   
| ----------------| ------------------- |----------------- |--------------|-------------- |  
| 2104            | 5                   | 1                | 45           | 460           |  
| 1416            | 3                   | 2                | 40           | 232           |  
| 852             | 2                   | 1                | 35           | 178           |  

You will build a linear regression model using these values so you can then predict the price for other houses. For example, a house with 1200 sqft, 3 bedrooms, 1 floor, 40 years old.  

Please run the following code cell to create your `X_train` and `y_train` variables.

In [6]:
x_train = np.array([[2104, 5, 1, 45],
                    [1416, 3, 2, 40],
                    [852, 2, 1, 35]])
y_train = np.array([460, 232, 178])

In [7]:
b_init = 785.1811367994083
w_init = np.array([0.39133535, 18.75376741, -53.36032453, -26.42131618])

<a name="toc_15456_3"></a>
# 3 Model Prediction With Multiple Variables
The model's prediction with multiple variables is given by the linear model:

$$ f_{\mathbf{w},b}(\mathbf{x}) =  w_0x_0 + w_1x_1 +... + w_{n-1}x_{n-1} + b \tag{1}$$
or in vector notation:
$$ f_{\mathbf{w},b}(\mathbf{x}) = \mathbf{w} \cdot \mathbf{x} + b  \tag{2} $$ 
where $\cdot$ is a vector `dot product`

To demonstrate the dot product, we will implement prediction using (1) and (2).

In [21]:
def predict_single_loop(x,w,b):
    n = x.shape[0]
    result = 0
    for i in range(n):
        result_i = x[i] * w[i]
        result = result + result_i
    result += b
    return result
    

In [22]:
predict = predict_single_loop(x_train[0,:],w_init,b_init)
print(f"Predict for first house is {predict}")
print(f"Real value is {y_train[0]}")

Predict for first house is 459.9999976194083
Real value is 460


<a name="toc_15456_4"></a>
# 4 Compute Cost With Multiple Variables
The equation for the cost function with multiple variables $J(\mathbf{w},b)$ is:
$$J(\mathbf{w},b) = \frac{1}{2m} \sum\limits_{i = 0}^{m-1} (f_{\mathbf{w},b}(\mathbf{x}^{(i)}) - y^{(i)})^2 \tag{3}$$ 
where:
$$ f_{\mathbf{w},b}(\mathbf{x}^{(i)}) = \mathbf{w} \cdot \mathbf{x}^{(i)} + b  \tag{4} $$ 


In contrast to previous labs, $\mathbf{w}$ and $\mathbf{x}^{(i)}$ are vectors rather than scalars supporting multiple features.

In [ ]:
def compute_cost(X,y,w,b):
        """
    compute cost
    Args:
      X (ndarray (m,n)): Data, m examples with n features
      y (ndarray (m,)) : target values
      w (ndarray (n,)) : model parameters  
      b (scalar)       : model parameter
      
    Returns:
      cost (scalar): cost
    """
    m = X.shape[0]
    cost = 0.0
    for i in range(m):
        f_wb_i = np.dot(X[i],w) + b
        err = (f_wb_i - y[i])**2
        cost += err
    return cost / (2*m)

In [31]:
def compute_cost_vectoral(X,y,w,b):
    m = X.shape[0]
    f_wb = np.dot(X,w)+b  # 3 elemanlı bir array oluştu içinde predict değerleri var
    cost = np.sum((f_wb-y)**2 / (2*m)) # iki array arasında eleman elemana çıkarma işlemi
    return cost

In [33]:
cost = compute_cost_vectoral(x_train,y_train,w_init,b_init)
cost

np.float64(1.5578904045996676e-12)

<a name="toc_15456_5"></a>
# 5 Gradient Descent With Multiple Variables
Gradient descent for multiple variables:

$$\begin{align*} \text{repeat}&\text{ until convergence:} \; \lbrace \newline\;
& w_j = w_j -  \alpha \frac{\partial J(\mathbf{w},b)}{\partial w_j} \tag{5}  \; & \text{for j = 0..n-1}\newline
&b\ \ = b -  \alpha \frac{\partial J(\mathbf{w},b)}{\partial b}  \newline \rbrace
\end{align*}$$

where, n is the number of features, parameters $w_j$,  $b$, are updated simultaneously and where  

$$
\begin{align}
\frac{\partial J(\mathbf{w},b)}{\partial w_j}  &= \frac{1}{m} \sum\limits_{i = 0}^{m-1} (f_{\mathbf{w},b}(\mathbf{x}^{(i)}) - y^{(i)})x_{j}^{(i)} \tag{6}  \\
\frac{\partial J(\mathbf{w},b)}{\partial b}  &= \frac{1}{m} \sum\limits_{i = 0}^{m-1} (f_{\mathbf{w},b}(\mathbf{x}^{(i)}) - y^{(i)}) \tag{7}
\end{align}
$$
* m is the number of training examples in the data set

    
*  $f_{\mathbf{w},b}(\mathbf{x}^{(i)})$ is the model's prediction, while $y^{(i)}$ is the target value




### 📝 NOTE: Matrix Shapes in Vectorized Gradient Computation

**1. The $X^T$ Matrix (Transposed Feature Matrix)**
$$X^T = \begin{bmatrix} 
x_0^{(1)} & x_0^{(2)} & x_0^{(3)} \\ 
x_1^{(1)} & x_1^{(2)} & x_1^{(3)} \\ 
x_2^{(1)} & x_2^{(2)} & x_2^{(3)} \\ 
x_3^{(1)} & x_3^{(2)} & x_3^{(3)} 
\end{bmatrix}$$
$$\text{Shape of } X^T \implies \mathbf{(n \times m)} \;\;\; \text{or} \;\;\; \mathbf{(4 \times 3)}$$

**2. The Error Vector**
$$\text{Error} = \begin{bmatrix} 
e^{(1)} \\ 
e^{(2)} \\ 
e^{(3)} 
\end{bmatrix}$$
$$\text{Shape of Error} \implies \mathbf{(m \times 1)} \;\;\; \text{or} \;\;\; \mathbf{(3 \times 1)}$$

**3. The Dot Product Alignment**
$$(n \times \mathbf{m}) \cdot (\mathbf{m} \times 1) = (n \times 1)$$
$$(4 \times \mathbf{3}) \cdot (\mathbf{3} \times 1) \implies \text{Resulting Shape: } \mathbf{(4 \times 1)}$$

**Final Gradient Vector:**
$$\vec{dj\_dw} = \frac{1}{m} \Big( X^T \cdot \text{Error} \Big)$$




$$\huge\frac{\partial J(\vec{w},b)}{\partial w_j} = \frac{1}{m} \sum_{i=1}^{m} \underbrace{\left( f_{\vec{w},b}(\vec{x}^{(i)}) - y^{(i)} \right)}_{\text{ERROR}} x_j^{(i)}$$

In [2]:
def compute_gradient_vector(X,y,w,b):
    m = x.shape[0]
    error = (np.dot(X,w)+b) - y
    dj_dw = (np.dot(X.T,error)) / m
    return 